# This notebook is for experimentation purpose to check what to use and hos everything is working

In [ ]:
#  A very normal ollama chat completion method
import os
import ollama
from dotenv import load_dotenv
load_dotenv()

client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.getenv('ollama_api_key')}
)

messages = [
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
]

for part in client.chat('gpt-oss:120b', messages=messages, stream=True):
  print(part['message']['content'], end='', flush=True)

**Short answer:** The sky looks blue because molecules and tiny particles in Earth’s atmosphere scatter short‑wavelength (blue‑violet) sunlight much more efficiently than they scatter the longer‑wavelength red and orange light. Our eyes are more sensitive to blue than violet, and the violet light is partly absorbed by the upper atmosphere, so the scattered light that reaches us appears blue.

---

### The physics behind the color

| Step | What happens | Why it matters |
|------|--------------|----------------|
| 1. Sunlight reaches Earth | Sunlight is a mixture of all visible wavelengths (≈ 400–700 nm). | The light entering the atmosphere contains roughly equal amounts of red, orange, yellow, green, blue, indigo, and violet. |
| 2. Light interacts with the atmosphere | Photons collide with gases (N₂, O₂) and very small particles. | The particles are **much smaller** than the wavelength of visible light (size ≪ λ). |
| 3. Rayleigh scattering | For particles much smaller than λ, the sca

In [ ]:
# this function helps to create strings from given df
import pandas as pd
from tqdm import tqdm
from typing import Dict, LiteralString, List
import json

def create_data_from_df(path: str, column_desc: Dict):

    df = pd.read_csv(path)
    string_list = []
    for row in tqdm(df.iterrows()):

        row_string = ''
        row = row[1]

        for col in df.columns:
            desc = column_desc[col]
            row_string += f'Column name: {col}\n Data: {row[col]} \n\n'.lower()
        string_list.append(row_string)

    return string_list

In [ ]:
# create strings from df by attaching all the column values w.r.t its rows along with its column names so that the llm knows what
# value is what and segregate also all the values are lower cases to keep it normalized
col_desc_path_list = ['./data/pharmaceutical_df_col_desc.json', './data/supplier_df_col_desc.json']
csv_path_list = ["./source/pharmaceuticals.csv", "./source/supplychain.csv"]
df_string_list = []
for index in range(len(col_desc_path_list)):
    with open(col_desc_path_list[index], "r", encoding="utf-8") as f:
        data = json.load(f)
    csv_path = csv_path_list[index]
    df_string = create_data_from_df(csv_path, data)
    df_string_list.append(df_string)

final_df_string_list = []

final_df_string_list.extend(df_string_list[0])
final_df_string_list.extend(df_string_list[1])

121it [00:00, 7260.32it/s]
5000it [00:00, 20302.18it/s]


In [ ]:
from docx import Document  

# Extracting text from docx file
def extract_text(filename):  
    doc = Document(filename)  
    full_text = []  
    for para in doc.paragraphs:  
        full_text.append(para.text.lower())  
    return '\n'.join(full_text)  

text = extract_text(r"C:\Users\ujjaw\OneDrive\Desktop\knwoledge_base\genai\Source\Doc 2.docx")  
docx_text = text.split(' \n \n')

In [31]:
docx_text

['Job Description for Java Developer ',
 'Position- Java Developer  \nOpening- 3 Experience– 4 to 6 Years ',
 'Job Description:  \nJava developer is responsible for compete end to end web development includes UI, Business and  \nData layer. Understand Architecture Requirements and ensure effective Design, Development,  \nValidation and Support activities. Adherence to the organizational guidelines and processes. Primary  \nfocus will be on the latest Java technology with J2EE, Spring, REST API’s. You will be working with other  \nengineers and developers on different layers of the infrastructure therefore, commitment to  \ncollaborative problem solving, next generation design, and creating quality products is essential.  \nSkills Required  \nCore Java, J2EE, Spring MVC, Spring REST APIs, Spring Security, JSP, Web application, MS SQL Server,  \nRedis, Oauth2, Angular, JQuery  \nResponsibilities:  \n Design and build advanced Web applications with Core Java, J2EE.  \n Work with outside d

In [ ]:
import base64

# tried using Ollama multimodal capabilities but was not able to do so because of timecontraints
def analyze_with_ollama(image_path, prompt, model="llava"):  
    """Query Ollama with an image and prompt."""  
    with open(image_path, "rb") as f:  
        img_b64 = base64.b64encode(f.read()).decode()  
    try:  
        response = ollama.chat(
            client,
            model=model,  
            messages=[  
                {"role": "system", "content": "You are an expert document analyst. extract all the text, all the text from the gievn image, all the text and all the data and just give me back the extracted text."},  
                {"role": "user", "content": prompt, "images": [img_b64]}  
            ],  
            options={"temperature": 0.3, "num_predict": 1000}  
        )  
        return response["message"]["content"]  
    except Exception as e:  
        return f"Error processing image: {e}"

In [104]:
from pypdf import PdfReader
path = r".\Source"
extracted_pdf_text_list = []
for filename in os.listdir(path):
    print(filename)
    if 'pdf' in filename:
        filepath = path+'\\'+filename
        reader = PdfReader(filepath)  

        # Extract text from all pages  
        full_text = ""  
        for page_num, page in enumerate(reader.pages, start=1):  
            text = page.extract_text(extraction_mode='layout')  
            full_text += f"\n--- Page {page_num} ---\n{text}\n".lower()
        extracted_pdf_text_list.append(full_text)

Doc 1.pdf
Doc 2.docx
Doc 3.pdf
Doc 4.pdf
Doc 5.pdf
pharmaceuticals.csv
supplychain.csv


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter



In [27]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


C:\Users\ujjaw\AppData\Local\Temp\ipykernel_9976\745101351.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\ujjaw\OneDrive\Desktop\knwoledge_base\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# create embeddings to CSV and excel files 
# Split each paragraph into chunks

text_strings_to_process = [docx_text, extracted_pdf_text_list]
def create_chunks(text_list: List, non_overlap_text: List):

    text_splitter_overlapping = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Max characters
    chunk_overlap=100,   # Overlap between consecutive chunks
    separators=["\n \n", ".", "!", "?", ",", " ", ""]
    )

    text_splitter_non_overlapping = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Max characters/tokens per chunk
    separators=["\n \n", ".", "!", "?", ",", " ", ""]
    )
    all_chunks = []
    for arr in text_list:
        for text in arr:
            chunks = text_splitter_overlapping.split_text(text)
            all_chunks.extend(chunks)
    
    for text in non_overlap_text:
        chunks = text_splitter_overlapping.split_text(text)
        all_chunks.extend(chunks)
    
    print(f"Total chunks created: {len(all_chunks)}")

    return all_chunks

all_chunks = create_chunks(text_strings_to_process, final_df_string_list)

Total chunks created: 11269


In [ ]:
# tried creating embeddings usingthe normal method it was taking time about 3 hours then had to stop and start with a different approach and used async
embedding_list = []
for chunks in tqdm(all_chunks):
    embeddings = embeddings_model.embed_documents(chunks)
    embedding_list.append(embeddings)

print(f"✅ Created {len(embeddings), len(embedding_list)} embeddings.")

  0%|          | 109/32245 [00:36<2:57:05,  3.02it/s]


KeyboardInterrupt: 

In [108]:
# used async programming to create multiple embeddings in a single time

In [36]:
import asyncio

async def embed_chunks_async(chunks, model_name="all-MiniLM-L6-v2", batch_size=500):
    embeddings_model = HuggingFaceEmbeddings(model_name=model_name)

    async def embed_single_chunk(chunk):
        # Simulate async embedding (run in executor to prevent blocking)
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, embeddings_model.embed_query, chunk)

    # Process in batches concurrently
    all_embeddings = []
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[i:i + batch_size]
        results = await asyncio.gather(*[embed_single_chunk(c) for c in batch])
        all_embeddings.extend(results)
    return all_embeddings

In [37]:
def run_async_main(main_func):
    try:
        # Works in normal Python scripts
        asyncio.run(main_func)
    except RuntimeError:
        # If event loop is already running (like in notebooks)
        loop = asyncio.get_event_loop()
        if loop.is_running():
            task = loop.create_task(main_func)
            return task
        else:
            loop.run_until_complete(main_func)

In [106]:
created_embeddings = await run_async_main(embed_chunks_async(all_chunks))

100%|██████████| 23/23 [07:26<00:00, 19.40s/it]


In [ ]:
# creation of vector store
from langchain.vectorstores import FAISS
text_embeddings = [(all_chunks[idz], created_embeddings[idz]) for idz in range(len(created_embeddings))]
vectorstore = FAISS.from_embeddings(embedding = embeddings_model, text_embeddings= text_embeddings)
vectorstore.save_local("genai_embedding_store")

In [40]:
new_embd = embeddings_model.embed_documents("What is the order qty handled by Ranjit?")

In [43]:
len(new_embd)

40

In [50]:
vectorstore.similarity_search_with_relevance_scores("What is the order qty handled by Ranjit?", k = 10)

[(Document(id='a09367ee-28cc-4e87-9170-7918517c020b', metadata={}, page_content='Name or ID of the marketing specialist or manager handling the order Data: Ranjith \n\nColumn name: type_of_order Column Description: Category or classification of the order (e'),
  np.float32(0.35562176)),
 (Document(id='8cf0ca79-8e5f-4514-abf9-f2d0c530bfc7', metadata={}, page_content='. Jayaram \n\nColumn name: type_of_order Column Description: Category or classification of the order (e.g., standard, custom) Data: EP \n\nColumn name: product_ Column Description: Name or identifier of the product being ordered Data: TacSim \n\nColumn name: qty Column Description: Quantity of the product ordered Data: 36 \n\nColumn name: opp_id Column Description: Opportunity ID from the CRM system (e.g., Salesforce) Data: 5018'),
  np.float32(0.107280076)),
 (Document(id='d12fd3a1-b40e-454f-8388-0504d4056f51', metadata={}, page_content='.0 \n\nColumn name: order_receipt_date Column Description: Date when the sales order w

In [74]:
new_vector_store = FAISS.load_local('genai_embedding_store_new', embeddings_model, allow_dangerous_deserialization=True)

In [114]:
vectorstore.index.ntotal

11269

In [118]:
retrievers = vectorstore.as_retriever(search_kwargs = {"k":500})

In [ ]:
results = d.get_relevant_documents("What is the order qty handled by ranjit?".lower())

In [111]:
def keyword_filter(doc, keywords):
    return any(kw.lower() in doc.page_content.lower() for kw in keywords)

hybrid_results = [doc for doc in d.get_relevant_documents(query)
                  if keyword_filter(doc, "ranjit")]

In [113]:
len(hybrid_results)

100

In [119]:
def chat_completion(user_query, context):
    client = ollama.Client(
        host="https://ollama.com",
        headers={'Authorization': 'Bearer ' + os.getenv('ollama_api_key')}
    )

    prompt = " You are a very good question asnwering bot, You can easily find the answers to the questions asked \
                    You will be given a CONTEXT and USER QUERY you need to find answer from the given context and finally respond"
    

    master_prompt = "<USERY QUERY>" + user_query + '</USER QUERY>\n\n<CONTEXT>' + str(context) + '</CONTEXT>'

    message = [
        {
            "role":'system', 'content': prompt
        },
        {
            "role":"user", "content": master_prompt
        }
    ]
    response = client.chat('gpt-oss:120b', messages=message)['message'].content
    return response


In [116]:
questions = [
    "What is the order qty handled by Ranjit?",
    "What is the percentage of orders that haven't dispatched?",
    "List of products under the recoil kits orders?",
    "How much GST was charged for my insurance and when does my third party insurance expire?",
    "What are the roles we are currently hiring for?",
    "For the product Glock 17 what is the planned WO release date?",
    "Ok try Glock - 17",
    "What is our criteria to hire a data scientist?",
    "What are the benifits of log book and how can i set it up?",
    "How do i create a zone?"
]


In [117]:
responses = []

In [120]:
for ques in tqdm(questions):
    context = retrievers.get_relevant_documents(ques.lower())
    context = [value.page_content for value in context]
    response = chat_completion(ques.lower(), context)
    responses.append(response)

100%|██████████| 10/10 [02:33<00:00, 15.35s/it]


In [ ]:
extracted_data_against_questions = dict(zip(questions, responses))

{'What is the order qty handled by Ranjit?': 'The record with **mktg_specialistsmanagers =\u202f“ranjit”** shows a quantity of **1**.',
 "What is the percentage of orders that haven't dispatched?": 'Based on the data provided, there are **four** records whose status is\u202f*not\u202fdispatched* while the vast majority of the “order‑to‑dispatch” records have a status of\u202f*dispatched*.\u202fThat means only a small handful of the total orders are still pending.\n\nTaking the four “not\u202fdispatched” rows against the total number of rows in the\u202f*order‑to‑dispatch*\u202fset (dispatched\u202f+\u202fnot\u202fdispatched), the share of orders that haven’t been dispatched works out to roughly **2\u202f%–4\u202f%** – about **3\u202f%** of the orders.',
 'List of products under the recoil kits orders?': '**Products that appear in orders with\u202f`type_of_order = recoil kits`**\n\n- 7.62\u202fmm\u202fSLR  \n- Rifle\u202f5.56\u202f×\u202f45\u202fmm,\u202fX‑95\u202fTavor  \n- Glock\u202f

In [124]:
extracted_data_against_questions = dict(zip(questions, responses))

In [126]:
df = pd.DataFrame(extracted_data_against_questions, columns = ['Questions', 'Answers'])

In [129]:
df.to_excel("Generated response sheet.xlsx")

In [128]:
!pip install openpyxl


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
